In [1]:
import pandas as pd
from scipy.io import arff
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
import os
import numpy as np
from natsort import natsorted
import matplotlib.pyplot as plt

In [2]:
def realizar_undersampling(df,label_valor, quantidade):
    """
    Reduz a quantidade de amostras de uma label específica no dataset.
    
    Args:
        df (pd.DataFrame): O seu dataset original.
        coluna_label (str): O nome da coluna de alvo (target/label).
        label_valor: O valor da classe que você deseja reduzir.
        quantidade (int): O número exato de amostras que você quer manter dessa classe.
        
    Returns:
        pd.DataFrame: Dataset combinado com a label reduzida e as outras classes intactas.
    """
    
    # Separar os dados da label alvo e os dados das demais labels
    df_alvo = df[df["class"] == label_valor]
    df_resto = df[df["class"] != label_valor]
    
    # Validação simples para evitar erros caso a quantidade pedida seja maior que a existente
    if quantidade > len(df_alvo):
        print(f"⚠️ Aviso: A quantidade solicitada ({quantidade}) é maior que o disponível ({len(df_alvo)}).")
        print("Retornando o dataset original sem alterações na label.")
        return df

    # Realizar a amostragem aleatória (undersampling)
    df_alvo_reduzido = df_alvo.sample(n=quantidade, random_state=42)
    
    # Concatenar de volta com o restante dos dados
    df_final = pd.concat([df_alvo_reduzido, df_resto])
    
    # Embaralhar o dataset final para evitar que as classes fiquem ordenadas
    return df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
#obtendo datasets a serem testados
dir = "/home/alexandreselani/Desktop/Eucalyptus_preliminar/hc-features/"
files = natsorted(os.listdir(dir))


#inicializando dicionarios
per_class_results = {} #armazena metricas por classe
results = {file:{"acc":[],"f1":[]} for file in files} #armazena metricas gerais 

for features in files:
    print(f"DATASET: {features} ---------------")

    path = os.path.join(dir,features)
    print(path)
    data, meta = arff.loadarff(path)
    dataset = pd.DataFrame(data)

    dataset['class'] = dataset['class'].astype(int) #preprocessamento (nao eh importante)

    
    #dataset = realizar_undersampling(dataset,1,195)
    #dataset = realizar_undersampling(dataset,2,195)

    print(dataset["class"].value_counts())

    X = dataset.drop("class",axis=1)
    y = dataset.iloc[:,-1]

    

    per_class_results[features] = {}
    kfold = StratifiedKFold(random_state=42,shuffle=True)

    cm = np.zeros((len(dataset["class"].unique()),len(dataset["class"].unique())),dtype=int)
    print(cm.shape)
    #4 folds used in test and 1 in training as described by Mailson
    for i, (train_index, test_index) in enumerate(kfold.split(X, y)):
        #treinamento e teste
        X_train = X.iloc[train_index]
        y_train = y.iloc[train_index]

        X_test = X.iloc[test_index]
        y_test = y.iloc[test_index]
        
        scaler = StandardScaler()
        scaler = scaler.fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        model = SVC(kernel='rbf', C=1,random_state=42)
        model.fit(X_train, y_train)

        predicts = model.predict(X_test)

        #avaliacao geral
        accuracy = accuracy_score(y_test,predicts)
        f1 = f1_score(y_test,predicts,average="weighted")

        results[features]["acc"].append(accuracy)
        results[features]["f1"].append(f1) 

        #avaliacao por classe
        report = classification_report(y_test,predicts,output_dict=True,zero_division=0)
        
        for label, metrics in report.items():
            if label in ['accuracy', 'macro avg', 'weighted avg']:
                continue
            
            if label not in per_class_results[features]:
                per_class_results[features][label] = {"precision": [], "recall": [], "f1-score": []}
            
            per_class_results[features][label]["precision"].append(metrics["precision"])
            per_class_results[features][label]["recall"].append(metrics["recall"])
            per_class_results[features][label]["f1-score"].append(metrics["f1-score"])

        #matriz de confusao
        cm += confusion_matrix(y_test,predicts)

    #plottando matrizes de confusao
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=model.classes_)
    disp.plot(ax=ax)
    plt.title(f"Confusion matrix - hc features: {features}")
    image_name = f"cm_{features.replace('.arff', '')}.png"
    plt.savefig(image_name, dpi=300, bbox_inches='tight')
    plt.close(fig)

#escrevendo metricas por classe
final_data = []

for filename, classes in per_class_results.items():
    for label, metrics in classes.items():
        final_data.append({
            "Dataset": filename,
            "Class": label,
            "Prec_Mean": np.mean(metrics["precision"]),
            "Prec_Std": np.std(metrics["precision"]),
            "Recall_Mean": np.mean(metrics["recall"]),
            "Recall_Std": np.std(metrics["recall"]),
            "F1_Mean": np.mean(metrics["f1-score"]),
            "F1_Std": np.std(metrics["f1-score"])
        })
    
    final_data.append({
            "Dataset": "",
            "Class": "",
            "Prec_Mean": np.nan,
            "Recall_Mean": np.nan,
            "F1_Mean": np.nan,
            "F1_Std": np.nan
        })

df_per_class_results = pd.DataFrame(final_data)
df_per_class_results.to_csv("metricas_por_classe_svm.csv", index=False, float_format="%.3f")

#escrevendo metricas gerais
final_data = []

for filename, metrics in (results.items()):
    final_data.append( row = {
                    "f1_macro_mean": np.mean(metrics['f1']),
                    "f1_macro_std": np.std(metrics['f1']),
                    "acc_mean": np.mean(metrics['acc']),
                    "acc_std": np.std(metrics['acc']),
                    "uuc_acc_mean": np.mean(metrics['uuc_acc']),
                    "uuc_acc_std": np.std(metrics['uuc_acc']),
                    "inner_mean": np.mean(metrics['inner']),
                    "inner std": np.std(metrics["inner"]),
                    "outer_mean": np.mean(metrics['outer']),
                    "outer_std": np.std(metrics["outer"]),
                    "halfpoint_mean": np.mean(metrics['half'][0]),
                    "halfpoint_std": np.std(metrics['half']),
                    "auroc_mean": np.mean(metrics['auroc']),
                    "auroc_std": np.std(metrics['auroc'])
                })

# Criar DataFrame e salvar em CSV
df_results = pd.DataFrame(final_data)
df_results.to_csv("resultado_final_svm.csv", index=False, float_format="%.3f") 

DATASET: ceratocystis1.arff ---------------
./hc-features/ceratocystis1.arff
class
1    247
2    247
3    195
Name: count, dtype: int64
(3, 3)
DATASET: ceratocystis2.arff ---------------
./hc-features/ceratocystis2.arff
class
1    494
2    494
3    195
Name: count, dtype: int64
(3, 3)
DATASET: ceratocystis5.arff ---------------
./hc-features/ceratocystis5.arff
class
1    1236
2    1236
3     195
Name: count, dtype: int64
(3, 3)
DATASET: ceratocystis10.arff ---------------
./hc-features/ceratocystis10.arff
class
2    2473
1    1402
3     195
Name: count, dtype: int64
(3, 3)
DATASET: ceratocystis20.arff ---------------
./hc-features/ceratocystis20.arff
class
2    4946
1    1402
3     195
Name: count, dtype: int64
(3, 3)
DATASET: dataset1_index_features.csv ---------------
./hc-features/dataset1_index_features.csv


StopIteration: 